In [7]:
import pandas as pd

# Load the CSV file
file_path = "cisco.csv"  # Change this if needed
df = pd.read_csv(file_path)

# Display the first few rows
df.head()


,Cost_Rank,Product_Name,Product_Life_Cycle,FY22_Q2,FY22_Q3,FY22_Q4,FY23_Q1,FY23_Q2,FY23_Q3,FY23_Q4,...,M_FY2025_Q1_BIAS,S_FY2024_Q3_Accuracy,S_FY2024_Q3_BIAS,S_FY2024_Q4_Accuracy,S_FY2024_Q4_BIAS,S_FY2025_Q1_Accuracy,S_FY2025_Q1_BIAS,D_FY2025_Q2_Forecasted,M_FY2025_Q2_Forecasted,S_FY2025_Q2_Forecasted
0,1,SWITCH Enterprise High,Sustaining,57147.0,52873.0,52870.0,38833.0,27114.0,21823,31813,...,70.11%,76.51%,23.49%,94.26%,-5.74%,84.94%,15.06%,29814,28378,26648
1,2,SWITCH Enterprise Ultra High,Sustaining,222.0,1549.0,4619.0,4764.0,5015.0,6656,9605,...,9.46%,93.05%,6.95%,46.30%,-53.70%,66.61%,-33.39%,11046,7748,11865
2,3,SWITCH Enterprise Ultra High,Sustaining,24362.0,21308.0,19067.0,14551.0,13271.0,10165,10477,...,81.29%,93.79%,-6.21%,8.94%,-91.06%,77.93%,-22.07%,10450,10785,9165
3,4,SWITCH Enterprise Low,Sustaining,NaN,NaN,1227.0,24186.0,7680.0,16772,17554,...,37.29%,78.81%,-21.19%,72.99%,-27.01%,48.12%,-51.88%,24505,29567,24186
4,5,TRANSCEIVER MODULE Mid,Sustaining,208760.0,116126.0,150803.0,82163.0,82408.0,67132,87498,...,42.22%,99.28%,-0.72%,89.31%,-10.69%,61.25%,38.75%,70000,71000,68858


In [9]:
# Convert percentage strings to numerical values for accuracy and bias columns
percentage_columns = [col for col in df.columns if "Accuracy" in col or "BIAS" in col]

# Remove '%' and convert to float
for col in percentage_columns:
    df[col] = df[col].str.rstrip('%').astype(float) / 100

# Check cleaned data
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 36 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Cost_Rank               20 non-null     int64  
 1   Product_Name            20 non-null     object 
 2   Product_Life_Cycle      20 non-null     object 
 3   FY22_Q2                 14 non-null     float64
 4   FY22_Q3                 14 non-null     float64
 5   FY22_Q4                 17 non-null     float64
 6   FY23_Q1                 17 non-null     float64
 7   FY23_Q2                 18 non-null     float64
 8   FY23_Q3                 20 non-null     int64  
 9   FY23_Q4                 20 non-null     int64  
 10  FY24_Q1                 20 non-null     int64  
 11  FY24_Q2                 20 non-null     int64  
 12  FY24_Q3                 20 non-null     int64  
 13  FY24_Q4                 20 non-null     int64  
 14  FY25_Q1                 20 non-null     int6

,Cost_Rank,Product_Name,Product_Life_Cycle,FY22_Q2,FY22_Q3,FY22_Q4,FY23_Q1,FY23_Q2,FY23_Q3,FY23_Q4,...,M_FY2025_Q1_BIAS,S_FY2024_Q3_Accuracy,S_FY2024_Q3_BIAS,S_FY2024_Q4_Accuracy,S_FY2024_Q4_BIAS,S_FY2025_Q1_Accuracy,S_FY2025_Q1_BIAS,D_FY2025_Q2_Forecasted,M_FY2025_Q2_Forecasted,S_FY2025_Q2_Forecasted
0,1,SWITCH Enterprise High,Sustaining,57147.0,52873.0,52870.0,38833.0,27114.0,21823,31813,...,0.7011,0.7651,0.2349,0.9426,-0.0574,0.8494,0.1506,29814,28378,26648
1,2,SWITCH Enterprise Ultra High,Sustaining,222.0,1549.0,4619.0,4764.0,5015.0,6656,9605,...,0.0946,0.9305,0.0695,0.4630,-0.5370,0.6661,-0.3339,11046,7748,11865
2,3,SWITCH Enterprise Ultra High,Sustaining,24362.0,21308.0,19067.0,14551.0,13271.0,10165,10477,...,0.8129,0.9379,-0.0621,0.0894,-0.9106,0.7793,-0.2207,10450,10785,9165
3,4,SWITCH Enterprise Low,Sustaining,NaN,NaN,1227.0,24186.0,7680.0,16772,17554,...,0.3729,0.7881,-0.2119,0.7299,-0.2701,0.4812,-0.5188,24505,29567,24186
4,5,TRANSCEIVER MODULE Mid,Sustaining,208760.0,116126.0,150803.0,82163.0,82408.0,67132,87498,...,0.4222,0.9928,-0.0072,0.8931,-0.1069,0.6125,0.3875,70000,71000,68858


In [11]:
# Function to compute forecasted values based on actual sales and bias
def compute_forecasted_values(actual, bias):
    return actual * (1 + bias)

# Define the exact column names from your dataset
actual_quarters = ['FY24_Q3', 'FY24_Q4', 'FY25_Q1']
bias_quarters = ['FY2024_Q3', 'FY2024_Q4', 'FY2025_Q1']  # Adjust if needed

teams = ['D', 'M', 'S']

# Compute forecasted values for each team and each quarter
for team in teams:
    for i in range(len(actual_quarters)):
        actual_col = actual_quarters[i]  # Actual sales column
        bias_col = f"{team}_{bias_quarters[i]}_BIAS"  # Bias column for each team
        forecast_col = f"{team}_{actual_quarters[i]}_Forecasted"  # New column to store forecasted values

        # Check if both actual sales and bias columns exist before calculation
        if actual_col in df.columns and bias_col in df.columns:
            df[forecast_col] = compute_forecasted_values(df[actual_col], df[bias_col])
        else:
            print(f"Skipping {forecast_col} because {actual_col} or {bias_col} is missing.")

# Display the first few rows to verify forecasted values
df[[col for col in df.columns if 'Forecasted' in col]].head()


,D_FY2025_Q2_Forecasted,M_FY2025_Q2_Forecasted,S_FY2025_Q2_Forecasted,D_FY24_Q3_Forecasted,D_FY24_Q4_Forecasted,D_FY25_Q1_Forecasted,M_FY24_Q3_Forecasted,M_FY24_Q4_Forecasted,M_FY25_Q1_Forecasted,S_FY24_Q3_Forecasted,S_FY24_Q4_Forecasted,S_FY25_Q1_Forecasted
0,29814,28378,26648,25999.5300,29853.8812,29553.9972,27278.7680,38598.6308,41707.5698,26957.8670,27716.2104,28210.4108
1,11046,7748,11865,9823.5296,11605.5792,10979.4400,6919.5624,9790.5600,10579.3090,9180.5880,5811.5760,6437.8565
2,10450,10785,9165,10199.7972,10911.7236,10234.7082,10895.3220,15416.9184,16658.7381,8674.6371,960.3348,7160.9877
3,24505,29567,24186,25008.7012,26605.4800,22000.4352,25315.3474,39039.6940,44987.1872,19179.9897,16049.0412,15767.9616
4,70000,71000,68858,69002.3200,81098.6974,75000.0922,78999.2040,71001.9352,77501.3668,76286.7520,76289.4951,75610.4250


In [13]:
# Print all column names to check for mismatches
print(df.columns.tolist())


['Cost_Rank', 'Product_Name', 'Product_Life_Cycle', 'FY22_Q2', 'FY22_Q3', 'FY22_Q4', 'FY23_Q1', 'FY23_Q2', 'FY23_Q3', 'FY23_Q4', 'FY24_Q1', 'FY24_Q2', 'FY24_Q3', 'FY24_Q4', 'FY25_Q1', 'D_FY2024_Q3_Accuracy', 'D_FY2024_Q3_BIAS', 'D_FY2024_Q4_Accuracy', 'D_FY2024_Q4_BIAS', 'D_FY2025_Q1_Accuracy', 'D_FY2025_Q1_BIAS', 'M_FY2024_Q3_Accuracy', 'M_FY2024_Q3_BIAS', 'M_FY2024_Q4_Accuracy', 'M_FY2024_Q4_BIAS', 'M_FY2025_Q1_Accuracy', 'M_FY2025_Q1_BIAS', 'S_FY2024_Q3_Accuracy', 'S_FY2024_Q3_BIAS', 'S_FY2024_Q4_Accuracy', 'S_FY2024_Q4_BIAS', 'S_FY2025_Q1_Accuracy', 'S_FY2025_Q1_BIAS', 'D_FY2025_Q2_Forecasted', 'M_FY2025_Q2_Forecasted', 'S_FY2025_Q2_Forecasted', 'D_FY24_Q3_Forecasted', 'D_FY24_Q4_Forecasted', 'D_FY25_Q1_Forecasted', 'M_FY24_Q3_Forecasted', 'M_FY24_Q4_Forecasted', 'M_FY25_Q1_Forecasted', 'S_FY24_Q3_Forecasted', 'S_FY24_Q4_Forecasted', 'S_FY25_Q1_Forecasted']


In [15]:
df.to_csv("forecasted_values.csv", index=False)


In [134]:
# Dictionary to store the best team for each product
best_teams_per_product = {}

# Loop through each product (row-wise operation)
for idx, row in df.iterrows():
    team_accuracies = {
        "D": row[["D_FY2024_Q3_Accuracy", "D_FY2024_Q4_Accuracy", "D_FY2025_Q1_Accuracy"]].mean(),
        "M": row[["M_FY2024_Q3_Accuracy", "M_FY2024_Q4_Accuracy", "M_FY2025_Q1_Accuracy"]].mean(),
        "S": row[["S_FY2024_Q3_Accuracy", "S_FY2024_Q4_Accuracy", "S_FY2025_Q1_Accuracy"]].mean(),
    }
    
    # Find the team with the highest average accuracy
    best_team = max(team_accuracies, key=team_accuracies.get)
    
    # Store the best team and their accuracy
    best_teams_per_product[df.loc[idx, "Product_Name"]] = (best_team, team_accuracies[best_team])

# Print results
print("Best forecasting team per product:\n")
for product, (team, accuracy) in best_teams_per_product.items():
    print(f"{product}: Best Team = {team}, Accuracy = {accuracy:.4f}")


Best forecasting team per product:

SWITCH Enterprise High : Best Team = D, Accuracy = 0.8295
SWITCH Enterprise Ultra High : Best Team = D, Accuracy = 0.9225
SWITCH Enterprise Low: Best Team = D, Accuracy = 0.8113
TRANSCEIVER MODULE Mid: Best Team = D, Accuracy = 0.7055
POWER SUPPLY High: Best Team = D, Accuracy = 0.9441
TRANSCEIVER MODULE High: Best Team = S, Accuracy = 0.4813
ACCESS POINT Mid: Best Team = D, Accuracy = 0.9828
POWER SUPPLY Mid: Best Team = S, Accuracy = 0.9047
SERVER : Best Team = S, Accuracy = 0.7001
PROCESSOR : Best Team = D, Accuracy = 0.6194
SWITCH Data Center High: Best Team = D, Accuracy = 0.8612
ROUTER Enterprise Mid : Best Team = M, Accuracy = 0.8742
MEMORY : Best Team = D, Accuracy = 0.5997
SWITCH Data Center Mid: Best Team = S, Accuracy = 0.8040
ROUTER Enterprise Low : Best Team = D, Accuracy = 0.7593


In [136]:
# Initialize dictionaries to store results
best_team_per_product = {}

# Loop through each product
for idx, row in df.iterrows():
    # Compute average accuracy for each team
    avg_accuracy = {
        "D": row[["D_FY2024_Q3_Accuracy", "D_FY2024_Q4_Accuracy", "D_FY2025_Q1_Accuracy"]].mean(),
        "M": row[["M_FY2024_Q3_Accuracy", "M_FY2024_Q4_Accuracy", "M_FY2025_Q1_Accuracy"]].mean(),
        "S": row[["S_FY2024_Q3_Accuracy", "S_FY2024_Q4_Accuracy", "S_FY2025_Q1_Accuracy"]].mean(),
    }

    # Compute average bias for each team
    avg_bias = {
        "D": row[["D_FY2024_Q3_BIAS", "D_FY2024_Q4_BIAS", "D_FY2025_Q1_BIAS"]].mean(),
        "M": row[["M_FY2024_Q3_BIAS", "M_FY2024_Q4_BIAS", "M_FY2025_Q1_BIAS"]].mean(),
        "S": row[["S_FY2024_Q3_BIAS", "S_FY2024_Q4_BIAS", "S_FY2025_Q1_BIAS"]].mean(),
    }

    # Find the team with the highest accuracy
    best_team = max(avg_accuracy, key=avg_accuracy.get)
    
    # Store results
    best_team_per_product[row["Product_Name"]] = {
        "Best Team": best_team,
        "Accuracy": avg_accuracy[best_team],
        "Bias": avg_bias[best_team]
    }

# Convert to DataFrame for better readability
best_team_df = pd.DataFrame.from_dict(best_team_per_product, orient="index")

# Print best forecasting team for each product
print(best_team_df)


                              Best Team  Accuracy      Bias
SWITCH Enterprise High                D  0.829500  0.170500
SWITCH Enterprise Ultra High          D  0.922533  0.077467
SWITCH Enterprise Low                 D  0.811267 -0.030333
TRANSCEIVER MODULE Mid                D  0.705467 -0.294533
POWER SUPPLY High                     D  0.944067  0.029667
TRANSCEIVER MODULE High               S  0.481300 -0.518700
ACCESS POINT Mid                      D  0.982833 -0.006700
POWER SUPPLY Mid                      S  0.904667  0.060600
SERVER                                S  0.700133 -0.115933
PROCESSOR                             D  0.619367  0.040833
SWITCH Data Center High               D  0.861167 -0.138833
ROUTER Enterprise Mid                 M  0.874167 -0.125833
MEMORY                                D  0.599733  0.547767
SWITCH Data Center Mid                S  0.803967 -0.196033
ROUTER Enterprise Low                 D  0.759267 -0.240733
